<a href="https://colab.research.google.com/github/sasindipeiris/EN4554_deep_learning_for_vision/blob/main/220454P_a02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EN4554 Assignment on CNNs


Name : Peiris T.P.N.S

Index number : 220454P

## 1. Setup

Use CIFAR-10 (torchvision.datasets.CIFAR10): split the training set into 45k
train and 5k validation, normalize per channel with training statistics, and fix random seeds.



In [5]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader, Subset
import numpy as np
import random

In [2]:
# Fix Random seeds for Reproducibility
def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

  set_seed(42)


In [7]:
raw_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)

# Split the training set into 45k train and 5k validation
generator = torch.Generator().manual_seed(42)
train_split, val_split = random_split(raw_dataset, [45000,5000],generator=generator)

# Calculate per-channel mean and std using only the training statistics
train_data = raw_dataset.data[train_split.indices] /255.0
mean_stats= train_data.mean(axis=(0,1,2))
std_stats = train_data.std(axis=(0,1,2))

print(f"Mean: {mean_stats} \nStd: {std_stats}")

# Normalize per channel with training statistics
base_transform = transforms.Compose([transforms.ToTensor(),
                                    transforms.Normalize(mean=mean_stats.tolist(),std=std_stats.tolist())])

# Instantiate seperate datasets for train and val
full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,transform=base_transform)
full_val_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,transform=base_transform)

train_subset = Subset(full_train_dataset, train_split.indices)
val_subset = Subset(full_val_dataset, val_split.indices)

train_loader= DataLoader(train_subset, batch_size=128, shuffle=True,num_workers=2)
val_loader = DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=2)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=base_transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)






Mean: [0.49165571 0.48233405 0.44667923] 
Std: [0.24708116 0.24350869 0.26158066]


## 2. Backpropagation

a. Implement Linear, ReLU, and softmax cross-entropy with explicit forward/backward (no autograd), and
assemble an MLP.

In [ ]:
import torch.nn.functional as F

class Linear:
  def __init__(self, in_features, out_features, seed=42):
    torch.manual_seed(seed)
    # Kaiming initialization
    self.W = torch.randn(in_features, out_features) * (2.0/in_features)**0.5
    self.b = torch.zeros(out_features)

    self.X = None
    self.grad_W = None
    self.grad_b = None

  def forward(self,X):
    self.X = X
    return X @ self.W + self.b

  def backward(self, dZ):
    self.grad_W = self.X.T @ dZ
    self.grad_b = dZ.sum(dim=0)
    dX = dZ @ self.W.T
    return dX





In [ ]:
class ReLU:
  def __init__(self):
    self.mask = None

  def forward(self,Z):
    self.mask = (Z>0).float()
    return Z*self.mask

  def backward(self,dA):
    return dA*self.mask

In [ ]:
class SoftmaxCrossEntropy:
    def __init__(self):
        self.probs = None
        self.targets = None

    def forward(self, logits, targets):
        # targets: (batch_size,) integer labels
        self.targets = targets

        # Shift logits for numerical stability
        logits_max = torch.max(logits, dim=1, keepdim=True).values
        exp_logits = torch.exp(logits - logits_max)
        self.probs = exp_logits / torch.sum(exp_logits, dim=1, keepdim=True)

        batch_size = logits.size(0)
        core_probs = self.probs[torch.arange(batch_size), targets]
        loss = -torch.log(core_probs + 1e-9).mean()
        return loss

    def backward(self):
        batch_size = self.probs.size(0)
        dZ = self.probs.clone()
        dZ[torch.arange(batch_size), self.targets] -= 1.0
        dZ = dZ /

In [ ]:
class CustomedMLP:
  def __init__(self, in_dim=3072, hidden_dim=256, out_dim=10):
    self.fc1 = Linear(in_dim, hidden_dim)
    self.relu = ReLU()
    self.fc2 = Linear(hidden_dim, out_dim)
    self.criterion = SoftmaxCrossEntropy()

  def forwad(self,X):
    Z1 = self.fc1.forward(X)
    A1 = self.relu.forward(Z1)
    logits = self.fc2.forward(A1)\
    return logits

  def backward(self):
    d_logits = self.criterion.backward()
    d_A1 = self.fc2.backward(d_logits)
    d_Z1 = self.relu.backward(d_A1)
    self.fc1.backward(d_Z1)

b. Verify your gradients against autograd and against centred finite differences; report the relative errors. Use
torch.allclose to check whether the gradients match